<a href="https://colab.research.google.com/github/Sakshiiikashyap/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This notebook audits the methodology behind two FlyRank research-paper findings and then stress-tests my Week-5 refresh/content opportunity model under a more conservative validation design. The goal is trustworthy, directional decision-support evidence rather than causal claims.

## 1. Two paper findings + my methodology questions

**Finding 1 — Content Lifecycle / Growing vs Declining:** The paper reports that growing pages are younger on average than declining pages (185 days vs 228 days), while average word count is almost the same (1,487 vs 1,481 words). My methodology question is: **how exactly is the growing/declining label defined and over what measurement window is it assigned?** Because the finding compares groups defined by traffic direction, I would want the label window and threshold disclosed clearly enough to know whether the comparison is measuring a short-term movement or a broader lifecycle pattern.

**Finding 2 — High Search Volume = More Traffic (tested as OPPOSITE):** The paper reports that search-volume estimates have effectively zero correlation with actual page traffic and argues that search volume is better treated as a competition/demand signal than a page-level traffic ceiling. My methodology question is: **does the validation design make the comparison genuinely comparable across measurement windows and page/query matching?** I would want to confirm how third-party search volume is aligned with page-level impressions, why the 90-day impressions are divided by three, and whether the result is intended as a directional portfolio-level relationship rather than a causal statement.

These are constructive review questions: they do not reject the findings; they identify the definitions and validation choices that a reader would need in order to judge how far the evidence can support the wording.

In [12]:
import pandas as pd

# Section 1 check: record the two findings and methodology questions as structured text.
paper_audit = pd.DataFrame([
    {
        "Finding": "Growing pages are younger than declining pages; word count is nearly the same.",
        "Methodology question": "How is the growing/declining label defined, and what measurement window and threshold create that label?",
    },
    {
        "Finding": "High search volume does not reliably predict page traffic in this portfolio.",
        "Methodology question": "How are search-volume estimates aligned with page-level traffic windows and page/query matching, and is the result directional rather than causal?",
    },
])

display(paper_audit)

,Finding,Methodology question
0,Growing pages are younger than declining pages...,"How is the growing/declining label defined, an..."
1,High search volume does not reliably predict p...,How are search-volume estimates aligned with p...


## 2. My model under an honest split (before/after)

I compare two validation designs using the same features, Random Forest configuration, target, and Precision@50 metric. **Before:** a stratified row split, which can place pages from the same client in both train and test. **After:** a client-holdout split, where all pages from selected clients are kept out of training. The client holdout is the more conservative design for this dataset because related pages from the same client should not cross the train/test boundary.

In [13]:
import os

print(os.listdir("/content"))


['.config', 'flyrank_project', 'flyrank-ml-internship-main.zip', 'sample_data']


In [14]:
import zipfile
from pathlib import Path

ZIP_PATH = Path("/content/flyrank-ml-internship-main.zip")
EXTRACT_PATH = Path("/content/flyrank_project")

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("✅ ZIP extracted!")
print("CSV files found:")

for p in EXTRACT_PATH.rglob("content_refresh_anonymized.csv"):
    print(p)

✅ ZIP extracted!
CSV files found:
/content/flyrank_project/flyrank-ml-internship-main/data/raw/content_refresh_anonymized.csv


In [15]:
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42

# Locate the CSV whether the notebook is run from the repo or from Colab with the ZIP uploaded.
csv_name = "content_refresh_anonymized.csv"
candidate_paths = [
    Path.cwd() / "data" / "raw" / csv_name,
    Path("/content/flyrank_project") / "data" / "raw" / csv_name,
    Path("/content/flyrank-ml-internship-main") / "data" / "raw" / csv_name,
]

DATA_PATH = next((p for p in candidate_paths if p.exists()), None)

if DATA_PATH is None:
    zip_candidates = list(Path("/content").glob("*.zip"))
    if zip_candidates:
        extract_path = Path("/content/flyrank_project")
        extract_path.mkdir(exist_ok=True)
        with zipfile.ZipFile(zip_candidates[0], "r") as z:
            z.extractall(extract_path)
        matches = list(extract_path.rglob(csv_name))
        if matches:
            DATA_PATH = matches[0]

if DATA_PATH is None:
    raise FileNotFoundError("Could not locate content_refresh_anonymized.csv.")

df = pd.read_csv(DATA_PATH)
df["is_declining_label"] = df["trend_direction"].astype(str).str.lower().eq("down").astype(int)

FEATURES = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
]

X = df[FEATURES].replace([np.inf, -np.inf], np.nan)
y = df["is_declining_label"].astype(int)

def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    k = min(k, len(y_true))
    order = np.argsort(-scores, kind="mergesort")[:k]
    return float(y_true[order].mean())

def fit_and_score(train_idx, test_idx):
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            min_samples_leaf=25,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    scores = model.predict_proba(X.iloc[test_idx])[:, 1]
    return model, scores, precision_at_k(y.iloc[test_idx], scores, 50)

all_idx = np.arange(len(df))

# BEFORE: stratified row split
before_train, before_test = train_test_split(
    all_idx,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)
before_model, before_scores, before_p50 = fit_and_score(before_train, before_test)

# AFTER: client-holdout split
client_series = df["client_id"].fillna("unknown").astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(unique_clients)
n_test_clients = max(1, int(round(len(shuffled_clients) * 0.20)))
test_clients = set(shuffled_clients[:n_test_clients])
after_test_mask = client_series.isin(test_clients).to_numpy()
after_train = all_idx[~after_test_mask]
after_test = all_idx[after_test_mask]

if y.iloc[after_train].nunique() < 2 or y.iloc[after_test].nunique() < 2:
    raise ValueError("Client holdout did not contain both target classes; inspect the client split before proceeding.")

after_model, after_scores, after_p50 = fit_and_score(after_train, after_test)

overlap = set(client_series.iloc[after_train]) & set(client_series.iloc[after_test])
assert len(overlap) == 0

comparison = pd.DataFrame([
    {"Validation": "Before — stratified row split", "Test rows": len(before_test), "Precision@50": before_p50},
    {"Validation": "After — client holdout", "Test rows": len(after_test), "Precision@50": after_p50},
])

display(comparison.round(3))
print(f"Before Precision@50: {before_p50:.3f}")
print(f"After Precision@50:  {after_p50:.3f}")
print(f"Change after honest split: {after_p50 - before_p50:+.3f}")
print(f"Client overlap after split: {len(overlap)}")

,Validation,Test rows,Precision@50
0,Before — stratified row split,6000,0.92
1,After — client holdout,2325,0.38


Before Precision@50: 0.920
After Precision@50:  0.380
Change after honest split: -0.540
Client overlap after split: 0


## 3. Leakage audit

The target is `trend_direction == "down"`. The model must not use `trend_direction` or `trend_pct`, because these are label-derived outcome information. I also exclude identifiers such as `content_id` and `client_id` from the model feature matrix; `client_id` is used only to create the grouped validation split.

In [16]:
forbidden_features = {"trend_direction", "trend_pct"}
identifier_features = {"content_id", "client_id"}

feature_set = set(FEATURES)
leakage_hits = feature_set & forbidden_features
identifier_hits = feature_set & identifier_features

print("Model features:", FEATURES)
print("Label-derived feature leakage:", leakage_hits)
print("Identifier features used as model inputs:", identifier_hits)

assert not leakage_hits, "Leakage detected: label-derived feature entered the model."
assert not identifier_hits, "Identifier entered the model feature set."
assert len(overlap) == 0, "Client overlap detected in honest split."

print("LEAKAGE AUDIT PASSED")

Model features: ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count']
Label-derived feature leakage: set()
Identifier features used as model inputs: set()
LEAKAGE AUDIT PASSED


## 4. Claim rewrite

**Overly strong version:** “The Random Forest model is better than the baseline and can identify the pages that will decline.”

**Evidence-aligned version:** “On the evaluated test split, the Random Forest produced a measured Precision@50 that can be compared with the Week-4 baseline. The client-holdout result is the more conservative estimate because pages from held-out clients were excluded from training. The model should therefore be treated as directional decision-support for prioritising review, not as proof that a page will decline or that a refresh will cause improvement.”

This wording separates what was **observed/measured** from what cannot be established causally by this validation design.

In [17]:
claim_audit = pd.DataFrame([
    {"Claim type": "Observed / measured", "Safe wording": f"Random Forest Precision@50 was {after_p50:.3f} under the client-holdout evaluation."},
    {"Claim type": "Directional", "Safe wording": "The model provides a directional ranking of pages that may deserve review."},
    {"Claim type": "Decision-support", "Safe wording": "The ranking can support a review queue, subject to human judgement and further validation."},
    {"Claim type": "Not established", "Safe wording": "This analysis does not establish that the model causes lower decline or that refreshing a page causes improved performance."},
])
display(claim_audit)

,Claim type,Safe wording
0,Observed / measured,Random Forest Precision@50 was 0.380 under the...
1,Directional,The model provides a directional ranking of pa...
2,Decision-support,"The ranking can support a review queue, subjec..."
3,Not established,This analysis does not establish that the mode...


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit the repo URL on the card. Done.